In [4]:
%pip install numpy pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


# Replay protection experiment analysis

This notebook reads the runner's raw observations and recomputes metrics rather than using fabricated sample results. Choose an experiment directory generated by `run-local-demo-2.js` in the first code cell. Latencies are runner-observed end-to-end HTTP timings; store latency is reported separately.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

experiment_dir = Path('../results/replay-2026-11-11T07-38-20-847Z-cdda86af').resolve()  # Set this to a generated run directory.
metadata = json.loads((experiment_dir / 'metadata.json').read_text())
requests = pd.read_csv(experiment_dir / 'requests.csv')
scenarios = pd.read_csv(experiment_dir / 'scenarios.csv')
summary = pd.read_csv(experiment_dir / 'summary.csv')

required_request_columns = {
    'experiment_id', 'scenario_id', 'method', 'replica_count', 'repetition',
    'phase', 'jti_hash', 'replica_id', 'latency_ms', 'actual_outcome',
    'replay_detected', 'legitimate_request', 'request_start_timestamp',
    'request_end_timestamp', 'store_latency_ms'
}
assert required_request_columns <= set(requests.columns)
assert {'scenario_id', 'status', 'completed_requests'} <= set(scenarios.columns)
assert {'scenario_id', 'latency_mean_ms', 'replay_detection_rate'} <= set(summary.columns)
metadata['experiment_id']

FileNotFoundError: [Errno 2] No such file or directory: '/home/nasyikhqusyairy/Projects/repositories/mse/results/<experiment-directory>/metadata.json'

In [ ]:
print('requests:', requests.shape, 'scenarios:', scenarios.shape, 'summary:', summary.shape)
display(pd.DataFrame({
    'requests': requests.isna().sum(),
    'scenarios': scenarios.isna().sum(),
    'summary': summary.isna().sum(),
}).fillna(0).astype(int))
display(scenarios)

## Security rates

Replay detection rate is replay rejections divided by replay attempts. In a concurrent batch, `n - 1` requests are replay attempts because one request is the expected first use. Legitimate acceptance rate is accepted legitimate requests divided by legitimate requests.

In [ ]:
requests['replay_detected'] = requests['replay_detected'].astype(str).str.lower().eq('true')
requests['legitimate_request'] = requests['legitimate_request'].astype(str).str.lower().eq('true')
security_rows = []
keys = ['scenario_id', 'method', 'replica_count', 'repetition']
for key, group in requests.groupby(keys, dropna=False):
    replay = group[group.phase == 'replay']
    concurrent = group[group.phase == 'concurrent_replay']
    # Exclude one expected first-use request per concurrent jti from the replay denominator.
    concurrent_replays = int(concurrent.groupby('jti_hash').size().sub(1).clip(lower=0).sum())
    replay_attempts = len(replay) + concurrent_replays
    concurrent_replay_rows = concurrent[~concurrent.legitimate_request]
    replay_blocked = int(replay.replay_detected.sum() + concurrent_replay_rows.replay_detected.sum())
    legitimate = group[group.legitimate_request]
    legitimate_accepted = int((legitimate.actual_outcome == 'accepted').sum())
    security_rows.append({
        **dict(zip(keys, key)),
        'replay_detection_rate': replay_blocked / replay_attempts if replay_attempts else np.nan,
        'legitimate_request_acceptance_rate': legitimate_accepted / len(legitimate) if len(legitimate) else np.nan,
        'false_acceptance_count': int((replay.actual_outcome == 'accepted').sum()) + max(0, int((concurrent.actual_outcome == 'accepted').sum()) - concurrent.jti_hash.nunique()),
        'cross_replica_replay_detection_rate': replay.loc[replay.replay_context == 'cross_replica', 'replay_detected'].mean(),
    })
security = pd.DataFrame(security_rows)
display(security)

## Latency and throughput

The latency distribution uses only raw `performance` requests. Mean, median, sample standard deviation (`ddof=1`), p95, and p99 use the request observations directly. Throughput is completed performance requests divided by the span between the earliest request start and latest request end within each concurrency window.

In [ ]:
requests['latency_ms'] = pd.to_numeric(requests['latency_ms'], errors='coerce')
requests['store_latency_ms'] = pd.to_numeric(requests['store_latency_ms'], errors='coerce')
requests['request_start_timestamp'] = pd.to_datetime(requests.request_start_timestamp, utc=True, errors='coerce')
requests['request_end_timestamp'] = pd.to_datetime(requests.request_end_timestamp, utc=True, errors='coerce')
performance = requests[requests.phase == 'performance'].copy()
latency_metrics = performance.groupby(keys, dropna=False).latency_ms.agg(
    latency_mean_ms='mean', latency_median_ms='median', latency_std_ms='std'
).reset_index()
percentiles = performance.groupby(keys, dropna=False).latency_ms.agg(
    latency_p95_ms=lambda values: values.quantile(.95, interpolation='linear'),
    latency_p99_ms=lambda values: values.quantile(.99, interpolation='linear'),
).reset_index()
latency_metrics = latency_metrics.merge(percentiles, on=keys, how='left')
display(latency_metrics)

throughput_rows = []
for group_key, group in performance.groupby(keys + ['concurrency_level'], dropna=False):
    group = group.dropna(subset=['request_start_timestamp', 'request_end_timestamp'])
    duration = (group.request_end_timestamp.max() - group.request_start_timestamp.min()).total_seconds() if len(group) else np.nan
    throughput_rows.append({
        **dict(zip(keys + ['concurrency_level'], group_key)),
        'completed_requests': len(group),
        'measurement_duration_seconds': duration,
        'throughput_rps': len(group) / duration if duration and duration > 0 else np.nan,
    })
throughput = pd.DataFrame(throughput_rows)
display(throughput)

In [ ]:
analysis_dir = experiment_dir / 'analysis'
analysis_dir.mkdir(exist_ok=True)
security.to_csv(analysis_dir / 'notebook_security_metrics.csv', index=False)
latency_metrics.to_csv(analysis_dir / 'notebook_latency_metrics.csv', index=False)
throughput.to_csv(analysis_dir / 'notebook_throughput.csv', index=False)

comparison = (security.groupby(['method', 'replica_count'], as_index=False).agg(
    replay_detection_rate=('replay_detection_rate', 'mean'),
    legitimate_request_acceptance_rate=('legitimate_request_acceptance_rate', 'mean'),
) if not security.empty else pd.DataFrame())
if not comparison.empty:
    comparison.pivot(index='method', columns='replica_count', values='replay_detection_rate').plot(
        kind='bar', ylim=(0, 1), rot=20, title='Replay detection by method and replica count'
    )
    plt.ylabel('Rate')
    plt.tight_layout()
    plt.savefig(analysis_dir / 'notebook_security_rates.png', dpi=150)
    plt.show()

if not performance.empty:
    performance.boxplot(column='latency_ms', by=['method', 'replica_count'])
    plt.suptitle('')
    plt.title('Runner-observed end-to-end latency')
    plt.ylabel('Latency (ms)')
    plt.tight_layout()
    plt.savefig(analysis_dir / 'notebook_latency.png', dpi=150)
    plt.show()

if not throughput.empty:
    for (method, replica_count), group in throughput.groupby(['method', 'replica_count']):
        group.groupby('concurrency_level').throughput_rps.mean().sort_index().plot(
            marker='o', label=f'{method}, {replica_count} replica(s)'
        )
    plt.title('Throughput by concurrency')
    plt.xlabel('Concurrency level')
    plt.ylabel('Requests per second')
    plt.legend()
    plt.tight_layout()
    plt.savefig(analysis_dir / 'notebook_throughput.png', dpi=150)
    plt.show()
display(comparison)